## Importing libraries

In [ ]:
import numpy as np
import pandas as pd
import re
from functions.functions import *

## Importing data

In [ ]:
df = pd.read_excel('../raw_data/Estrazioni_iconsulting.xlsx', sheet_name='Farmaceutica',
    dtype_backend='numpy_nullable', # For allowing the `string` dtype
    dtype={
        'PSEUDO_PAZ': 'string',
        'AIC_COD': 'string',
        # 'TIPO_EROGAZIONE': 'category' # Format 'TIPO_EROGAZIONE' column as categorical
    }
)

# Make the index start from 2 (as the original excel)
df.index = df.index + 2

## Parsing and reformatting

### `ATC_DES` column reformatting

In [ ]:
df = df.copy()
# Inconsistencies preprocessing
# # Pure inconsistency
df['ATC_DES'] = df['ATC_DES'].replace('METFORMINA ED EMPAGLIFLOZIN', 'METFORMINA E EMPAGLIFLOZIN')
df['ATC_DES'] = df['ATC_DES'].replace('ACIDO BEMPEDOICO ED EZETIMIBE', 'ACIDO BEMPEDOICO E EZETIMIBE')

# # `T.C. NON RILEVATO` values: leave as string...
# df['ATC_DES'] = df['ATC_DES'].replace('T.C. NON RILEVATO', 'TC_NON_RILEVATO')
# # ... or transform to NA
df['ATC_DES'] = df['ATC_DES'].replace('T.C. NON RILEVATO', pd.NA)

col = df['ATC_DES']

col = (
    col
    # case 3: "<a> <b> E <c>"   ->   "<a>_<b>+<c>"
    .str.replace(r'^(\S+)\s(\S+)\sE\s(\S+)$', r'\1_\2+\3', regex=True)

    # case 2: "<a> E <b>"   ->   "<a>+<b>"
    .str.replace(r'^(\S+)\sE\s(\S+)$', r'\1+\2', regex=True)

    # case 1: "<a> <b>"   ->   "<a>_<b>"
    .str.replace(r'^(\S+)\s(\S+)$', r'\1_\2', regex=True)
)

df['ATC_DES'] = col

### `AIC_DES` column reformatting

Original column can keeped decommenting line 43

In [ ]:
def split_column(value):
    """
    Requires pandas.
    Split a value with pattern `<str1>*<int><str2> <num><str3>`
    into three parts: `<str1>`, `<int><str2>`, `<num><str3>`
    If the row does not present a pattern like that, place a NA
    """
    # Check if value is NA/NaN/None
    if pd.isna(value):
        return pd.Series([pd.NA, pd.NA, pd.NA])

    # Split on '*' to get col1
    parts = value.split('*', 1)
    if len(parts) != 2:
        return pd.Series([pd.NA, pd.NA, pd.NA])
    
    col1 = parts[0]
    remainder = parts[1]
    
    # Find the last space followed by a number (handles comma as decimal separator)
    match = re.search(r'^(.+)\s+(\d+(?:,\d+)?.*?)$', remainder)
    
    if match:
        col2 = match.group(1)
        col3 = match.group(2)
    else:
        col2 = remainder
        col3 = None
    
    return pd.Series([col1, col2, col3])


# Get position and split
original_col_name = 'AIC_DES'
original_position = df.columns.get_loc(original_col_name)

# Split the column
split_df = df[original_col_name].apply(split_column)
split_df.columns = ['AIC_name', 'AIC_form', 'AIC_qty']
split_df = split_df.astype('string')

# Reconstruct dataframe: columns before + new columns + columns after
df = pd.concat([
    df.iloc[:, :original_position
        # +1 # Decomment for keeping the original column
    ],
    split_df,
    df.iloc[:, original_position+1:]
], axis=1)

### `AIC_name` column formatting
Choose if to keep brand names!

In [ ]:
def format_AIC_name(value, keep_brand=False):
    """
    Format AIC_name values based on number of whitespaces:
    - 0 whitespaces: one-word commercial name
        - leave unchanged
    - 1 whitespace: commercial name + brand
        - keep the brand (replace space with hyphen)
        - drop the brand
    - 2 whitespaces: two-word commercial name + brand
        - keep the brand (replace first space with underscore and second with hyphen)
        - drop the brand
    """
    # Handle NA values
    if pd.isna(value):
        return pd.NA
    
    # Count whitespaces
    whitespace_count = value.count(' ')
    
    if keep_brand:
        if whitespace_count == 0:
            return value
        elif whitespace_count == 1:
            return value.replace(' ', '-')
        elif whitespace_count == 2:
            parts = value.split(' ')
            return f"{parts[0]}_{parts[1]}-{parts[2]}"
        else:
            return value

    else:
        if whitespace_count == 0:
            return value
        elif whitespace_count == 1:
            parts = value.split(' ')
            return f"{parts[0]}"
        elif whitespace_count == 2:
            parts = value.split(' ')
            return f"{parts[0]}_{parts[1]}"
        else:
            return value

# Only drug with two different names
df['AIC_name'] = df['AIC_name'].replace('CLEXANE T', 'CLEXANE')

# Run the function
df['AIC_name'] = df['AIC_name'].apply(format_AIC_name, keep_brand=True).astype('string')

### Drop the only NA row?
**Pay attention to the label of the index!**

In [ ]:
# df = df.drop(1952)

### Reorder columns

In [ ]:
df = df[[
    'PSEUDO_PAZ' ,


# Data about drugs
    'AIC_COD' ,
    'AIC_DES' ,
    'ATC_COD' ,
    'ATC_DES' ,


# Unknown data
    'DDD_NDP_NUM' ,
    'DATA_SPEDIZIONE' ,
    'TIPO_EROGAZIONE' ,
]]

### Rename columns

In [ ]:
# df = df.rename(columns={
#     'original' : 'modified',
# })

### Drop columns with 50% or more NAs

In [ ]:
# df = drop_columns_by_na_threshold(df, threshold=0.5)

### Reorder by pseudopaz (and eventually reset index)

In [ ]:
df = df.sort_values(by=['PSEUDO_PAZ'])

In [ ]:
# df = df.reset_index(drop=True)

## Final checks

In [ ]:
df

In [ ]:
df.info()

## Export

In [ ]:
# # Parquet
# df.to_parquet("../raw_data/parquets/2_3-estr_icons-farma.parquet")

# # Pickle
# df.to_pickle("../raw_data/pickles/2_3-estr_icons-farma.pickle")

# # Excel
# df.to_excel('../raw_data/excel_formatted/2_3-estr_icons-farma.xlsx', index=False)